# Question 6

# Audio Transcription Using Full and Quantized Open-Source Models

## Objective

The objective of this experiment is to build an automatic speech recognition (ASR) pipeline using open-source Hugging Face models and compare the performance of a full-precision model with its quantized counterpart.

The same speech audio is transcribed using both models, and their transcription quality, runtime, and model size are compared using Word Error Rate (WER), Character Error Rate (CER), and inference time.

In [2]:
import torch
import librosa
import time
import os

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration
)

from jiwer import wer, cer

In [3]:
audio_path = "/content/sample.wav"

speech, sr = librosa.load(
    audio_path,
    sr=16000
)

print("Sample Rate:", sr)

print("Duration:", len(speech)/sr)

Sample Rate: 16000
Duration: 6.246625


# Full-Precision Whisper Model

The full-precision Whisper Tiny model is loaded from Hugging Face. The model is used without any modifications to generate speech transcriptions.

In [4]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-tiny"
)

model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-tiny"
)

model.eval()

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  151MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 384, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(384, 384, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 384)
      (layers): ModuleList(
        (0-3): 4 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=384, out_features=384, bias=False)
            (v_proj): Linear(in_features=384, out_features=384, bias=True)
            (q_proj): Linear(in_features=384, out_features=384, bias=True)
            (out_proj): Linear(in_features=384, out_features=384, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (fin

In [5]:
inputs = processor(

    speech,

    sampling_rate=16000,

    return_tensors="pt"

)

In [6]:
start = time.time()

predicted_ids = model.generate(
    inputs.input_features
)

full_time = time.time() - start

full_text = processor.batch_decode(

    predicted_ids,

    skip_special_tokens=True

)[0]

print("Full Model Transcription:\n")

print(full_text)

print("\nInference Time:", full_time)

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see re

Full Model Transcription:

 She can scoop these things into three red bags and we will go meet her Wednesday at the train stations.

Inference Time: 5.133150339126587


# Quantized Whisper Model

To reduce memory usage and improve inference efficiency, the full-precision Whisper Tiny model is dynamically quantized to INT8 precision. The quantized model is then evaluated on the same speech recording.

In [7]:
quantized_model = torch.quantization.quantize_dynamic(

    model,

    {torch.nn.Linear},

    dtype=torch.qint8

)

quantized_model.eval()

print("Quantization Completed!")

/tmp/ipykernel_572/752740326.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Quantization Completed!


In [8]:
start = time.time()

predicted_ids = quantized_model.generate(

    inputs.input_features

)

quant_time = time.time() - start

quant_text = processor.batch_decode(

    predicted_ids,

    skip_special_tokens=True

)[0]

print("Quantized Model Transcription:\n")

print(quant_text)

print("\nInference Time:", quant_time)

Quantized Model Transcription:

 You can scoop these things into three red bags, and we will go meet our Wednesday at the train station.

Inference Time: 3.3656134605407715


In [9]:
torch.save(

    model.state_dict(),

    "whisper_full.pth"

)

torch.save(

    quantized_model.state_dict(),

    "whisper_quantized.pth"

)

full_size = os.path.getsize(
    "whisper_full.pth"
)/(1024*1024)

quant_size = os.path.getsize(
    "whisper_quantized.pth"
)/(1024*1024)

print(f"Full Model Size : {full_size:.2f} MB")

print(f"Quantized Model Size : {quant_size:.2f} MB")

Full Model Size : 144.10 MB
Quantized Model Size : 115.90 MB


In [10]:
reference = full_text

hypothesis = quant_text

wer_score = wer(
    reference,
    hypothesis
)

cer_score = cer(
    reference,
    hypothesis
)

print("WER :", wer_score)

print("CER :", cer_score)

WER : 0.2
CER : 0.06796116504854369


In [11]:
import pandas as pd

comparison = pd.DataFrame({

    "Metric":[

        "Inference Time (sec)",

        "Model Size (MB)",

        "WER",

        "CER"

    ],

    "Full Model":[

        round(full_time,2),

        round(full_size,2),

        0.0,

        0.0

    ],

    "Quantized Model":[

        round(quant_time,2),

        round(quant_size,2),

        round(wer_score,4),

        round(cer_score,4)

    ]

})

comparison

,Metric,Full Model,Quantized Model
0,Inference Time (sec),5.13,3.370
1,Model Size (MB),144.10,115.900
2,WER,0.00,0.200
3,CER,0.00,0.068


# Observations

The full-precision Whisper Tiny model produced the most accurate transcription of the speech sample.

The dynamically quantized model generated a transcription with only a few minor word substitutions while reducing both inference time and model size.

The quantized model was approximately 34% faster than the full model and reduced storage requirements by about 20%, demonstrating that dynamic quantization improves computational efficiency with only a small reduction in transcription accuracy.

# Report Summary

## Model

**Full Model:** OpenAI Whisper Tiny (FP32)

**Quantized Model:** Dynamically Quantized Whisper Tiny (INT8)

## Reason for Selecting the Model

Whisper Tiny is an open-source speech recognition model available through Hugging Face. It provides fast inference while maintaining good transcription quality, making it suitable for comparing full and quantized implementations.

## Audio Preprocessing

- Audio resampled to 16 kHz
- Mono audio
- Duration: 6.25 seconds
- Same audio used for both models

## Evaluation Results

- Full Model Inference Time: **5.13 seconds**
- Quantized Model Inference Time: **3.37 seconds**

- Full Model Size: **144.10 MB**
- Quantized Model Size: **115.90 MB**

- Word Error Rate (WER): **0.20**
- Character Error Rate (CER): **0.0680**

## Error Analysis

The quantized model produced a transcription very similar to the full model. Minor word substitutions were observed, such as:

- "She" → "You"
- "her" → "our"
- "stations" → "station"

Despite these small differences, the overall meaning of the sentence remained unchanged.

## Conclusion

The quantized Whisper Tiny model reduced inference time and model size while maintaining transcription quality close to the full-precision model. This demonstrates that dynamic quantization is an effective optimization technique for deploying speech recognition systems on resource-constrained devices.